Run this notebook until Step 4 appears.

Before running the server cell, set NGROK_AUTHTOKEN in the notebook environment or Colab secrets. Do not hardcode the token in the notebook if you plan to push it to GitHub.

In [ ]:
import os
import subprocess

print("⚙️ Step 1: Installing system audio libraries (ffmpeg)...")
# Run this silently so it doesn't clutter the screen
subprocess.run("apt-get update && apt-get install -y ffmpeg", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("⚙️ Step 2: Installing AI and Web Tools (This takes ~1 minute)...")
# Removed transformers, bitsandbytes, and accelerate for a leaner install
subprocess.run("pip install -q git+https://github.com/m-bain/whisperx.git nest_asyncio fastapi uvicorn python-multipart pyngrok", shell=True)

print("⚙️ Step 3: Writing the Server Code...")
# The server code as a multi-line string
server_code = """
import os

# ==========================================
# 0. SERVER CONFIGURATION & API KEYS
# ==========================================
os.environ["MPLBACKEND"] = "Agg"

# 👉 Load your ngrok token from the environment before running this notebook.
ngrok_token = os.getenv("NGROK_AUTHTOKEN", "").strip()
if not ngrok_token:
    raise ValueError("NGROK_AUTHTOKEN is not set. Add it to the notebook environment before running the server.")
os.environ["NGROK_AUTHTOKEN"] = ngrok_token

import nest_asyncio
import uvicorn
import torch
import whisperx
from fastapi import FastAPI, UploadFile, File
from pyngrok import ngrok

app = FastAPI()

# ==========================================
# 1. MODEL LOADING
# ==========================================
print("Loading WhisperX... This will take a moment.")

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Loading WhisperX on {device} using {compute_type}...")
whisper_model = whisperx.load_model("small.en", device, compute_type=compute_type)

print("✅ WhisperX Loaded Successfully into VRAM!")

# ==========================================
# 2. FASTAPI ENDPOINTS
# ==========================================
@app.post("/evaluate-audio")
async def evaluate_audio(audio: UploadFile = File(...)):
    temp_audio_path = f"/tmp/{audio.filename}"

    try:
        # Save incoming audio
        with open(temp_audio_path, "wb") as buffer:
            buffer.write(await audio.read())

        # Transcribe with WhisperX
        audio_data = whisperx.load_audio(temp_audio_path)
        result = whisper_model.transcribe(audio_data, batch_size=16, language="en")
        transcript = " ".join([segment["text"] for segment in result["segments"]])

        return {
            "status": "success",
            "transcript": transcript
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

# ==========================================
# 3. SERVER EXECUTION
# ==========================================
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"] )
public_url = ngrok.connect(8000).public_url
print(f"\n>>> 🚀 YOUR API URL IS: {public_url}/evaluate-audio <<<\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)
"""

# Write the string to a file
with open("server.py", "w") as f:
    f.write(server_code)

print("🚀 Step 4: Booting up the server...\n")
# Run the newly created server file
os.system("python server.py")

after step 4 stop running the previous cell, keep this running from then on as long as the website is running

In [ ]:

!python server.py